# Census Income Prediction using PyTorch
This notebook demonstrates how to build a binary classification model using PyTorch to predict whether an individual earns more than $50,000 annually.

## 1. Data Preparation
Separate categorical, continuous, and label columns. Create arrays and tensors. Split into train and test sets.

In [15]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import gdown
import os
import warnings
warnings.filterwarnings('ignore')

In [16]:
# Download the dataset from Google Drive for Google Colab
file_id = '1ay5vOv2YiOwjKIWnXFT6sptW0gqew0oa'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'income.csv'

if not os.path.exists(output):
    print("Downloading dataset...")
    gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (30000, 10)


,age,sex,education,education-num,marital-status,workclass,occupation,hours-per-week,income,label
0,27,Male,HS-grad,9,Never-married,Private,Craft-repair,40,<=50K,0
1,47,Male,Masters,14,Married,Local-gov,Exec-managerial,50,>50K,1
2,59,Male,HS-grad,9,Divorced,Self-emp,Prof-specialty,20,<=50K,0
3,38,Female,Prof-school,15,Never-married,Federal-gov,Prof-specialty,57,>50K,1
4,64,Female,11th,7,Widowed,Private,Farming-fishing,40,<=50K,0


In [17]:
# Identify categorical, continuous, and label columns
# We assume the target column is the last column (typical for income datasets)
target_col = df.columns[-1]

# Convert target to 0 and 1 if it is categorical
if df[target_col].dtype == 'object':
    df[target_col] = LabelEncoder().fit_transform(df[target_col])

# Separate categorical and continuous columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cont_cols = df.select_dtypes(exclude=['object', 'category']).columns.tolist()

if target_col in cont_cols:
    cont_cols.remove(target_col)
if target_col in cat_cols:
    cat_cols.remove(target_col)

print(f"Categorical columns: {cat_cols}")
print(f"Continuous columns: {cont_cols}")

Categorical columns: ['sex', 'education', 'marital-status', 'workclass', 'occupation', 'income']
Continuous columns: ['age', 'education-num', 'hours-per-week']


In [18]:
# Convert categorical columns to category type and get their codes
for cat in cat_cols:
    df[cat] = df[cat].astype('category')

# Create embeddings sizes
cat_szs = [len(df[col].cat.categories) for col in cat_cols]
emb_szs = [(size, min(50, (size+1)//2)) for size in cat_szs]

# Create arrays for categorical values, continuous values, and labels
cats = np.stack([df[col].cat.codes.values for col in cat_cols], 1)
conts = np.stack([df[col].values for col in cont_cols], 1)

# Create tensors
cats = torch.tensor(cats, dtype=torch.int64)
conts = torch.tensor(conts, dtype=torch.float)
labels = torch.tensor(df[target_col].values, dtype=torch.long)

print(f"Categorical tensor shape: {cats.shape}")
print(f"Continuous tensor shape: {conts.shape}")
print(f"Labels tensor shape: {labels.shape}")

Categorical tensor shape: torch.Size([30000, 6])
Continuous tensor shape: torch.Size([30000, 3])
Labels tensor shape: torch.Size([30000])


In [19]:
# Split the dataset into training and testing sets (use 25,000 for training and 5,000 for testing)
b = 25000 # Training samples
t = 5000  # Testing samples

cat_train = cats[:b]
cat_test = cats[b:b+t]
cont_train = conts[:b]
cont_test = conts[b:b+t]
y_train = labels[:b]
y_test = labels[b:b+t]

print(f"Training set: {cat_train.shape[0]} samples")
print(f"Testing set: {cat_test.shape[0]} samples")

Training set: 25000 samples
Testing set: 5000 samples


## 2. Model Design
Define a TabularModel class and create a model with one hidden layer of 50 neurons and dropout p=0.4.

In [20]:
class TabularModel(nn.Module):
    def __init__(self, emb_szs, n_cont, out_sz, layers, p=0.5):
        super().__init__()

        # Embeddings for categorical features
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni, nf in emb_szs])
        self.emb_drop = nn.Dropout(p)

        # Batch normalization for continuous features
        self.bn_cont = nn.BatchNorm1d(n_cont)

        # Calculate total input size
        n_emb = sum((nf for ni, nf in emb_szs))
        n_in = n_emb + n_cont

        # Build layers
        layerlist = []
        for i in layers:
            layerlist.append(nn.Linear(n_in, i))
            layerlist.append(nn.ReLU(inplace=True))
            layerlist.append(nn.BatchNorm1d(i))
            layerlist.append(nn.Dropout(p))
            n_in = i
        layerlist.append(nn.Linear(layers[-1], out_sz))

        self.layers = nn.Sequential(*layerlist)

    def forward(self, x_cat, x_cont):
        embeddings = []
        for i, e in enumerate(self.embeds):
            embeddings.append(e(x_cat[:, i]))
        x = torch.cat(embeddings, 1)
        x = self.emb_drop(x)

        x_cont = self.bn_cont(x_cont)
        x = torch.cat([x, x_cont], 1)

        x = self.layers(x)
        return x

In [21]:
# Set a random seed for reproducibility
torch.manual_seed(42)

# Create the model: one hidden layer of 50 neurons and dropout p=0.4
model = TabularModel(emb_szs, conts.shape[1], 2, [50], p=0.4)
print(model)

TabularModel(
  (embeds): ModuleList(
    (0): Embedding(2, 1)
    (1): Embedding(14, 7)
    (2): Embedding(6, 3)
    (3): Embedding(5, 3)
    (4): Embedding(12, 6)
    (5): Embedding(2, 1)
  )
  (emb_drop): Dropout(p=0.4, inplace=False)
  (bn_cont): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (0): Linear(in_features=24, out_features=50, bias=True)
    (1): ReLU(inplace=True)
    (2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=50, out_features=2, bias=True)
  )
)


## 3. Training
Use CrossEntropyLoss as criterion and Adam optimizer (lr=0.001). Train for 300 epochs.

In [22]:
# Define criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model for 300 epochs
epochs = 300
losses = []

for i in range(epochs):
    i += 1
    y_pred = model(cat_train, cont_train)
    loss = criterion(y_pred, y_train)
    losses.append(loss.item())

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 30 == 0:
        print(f'Epoch {i:3} | Loss: {loss.item():.4f}')

Epoch  30 | Loss: 0.4729
Epoch  60 | Loss: 0.3762
Epoch  90 | Loss: 0.3240
Epoch 120 | Loss: 0.2941
Epoch 150 | Loss: 0.2612
Epoch 180 | Loss: 0.2388
Epoch 210 | Loss: 0.2227
Epoch 240 | Loss: 0.2075
Epoch 270 | Loss: 0.2022
Epoch 300 | Loss: 0.1827


## 4. Evaluation
Evaluate the model on the test set and report loss and accuracy.

In [23]:
# Evaluate the model on the test set
model.eval()
with torch.no_grad():
    y_val = model(cat_test, cont_test)
    loss = criterion(y_val, y_test)
    print(f'Test Loss: {loss.item():.4f}')

# Report accuracy
preds = torch.max(y_val, 1)[1]
correct = (preds == y_test).sum()
accuracy = 100 * correct / len(y_test)
print(f'Test Accuracy: {accuracy.item():.2f}%')

Test Loss: 0.1449
Test Accuracy: 96.54%


## 5. BONUS (Optional)
Write a function that allows a user to input new data and outputs the model’s prediction.

In [24]:
def predict_income(model, new_data_dict, df, cat_cols, cont_cols):
    """
    Function to predict income based on new user input.
    """
    model.eval()

    # Process categorical values
    cat_seq = []
    for col in cat_cols:
        val = new_data_dict.get(col, df[col].mode()[0]) # use mode if missing
        cats = df[col].cat.categories
        if val in cats:
            cat_seq.append(cats.get_loc(val))
        else:
            cat_seq.append(0) # fallback

    # Process continuous values
    cont_seq = []
    for col in cont_cols:
        val = new_data_dict.get(col, df[col].median()) # use median if missing
        cont_seq.append(val)

    # Convert to tensors
    cat_tensor = torch.tensor([cat_seq], dtype=torch.int64)
    cont_tensor = torch.tensor([cont_seq], dtype=torch.float)

    # Predict
    with torch.no_grad():
        z = model(cat_tensor, cont_tensor)
        pred = z.argmax(dim=1).item()

    return ">50K" if pred == 1 else "<=50K"

# Example user input
# You can modify these values to test different scenarios
sample_input = {
    'marital-status': 'Married-civ-spouse',
    'education': 'Bachelors',
    'hours-per-week': 45.0,
    'age': 35,
    'sex': 'Male'
}

prediction = predict_income(model, sample_input, df, cat_cols, cont_cols)
print(f"Predicted Income for the given input: {prediction}")

Predicted Income for the given input: <=50K
